# Garage Door Image Exploration

Goal: use low-quality Raspberry Pi camera images to infer whether the garage door is open or closed. The first pass here avoids a neural net and instead extracts robust brightness summaries from hand-picked regions of interest (ROIs). If the simple features overlap too much, the same feature table can feed a lightweight supervised classifier later.

Workflow:

1. Load images and timestamps from `data_path`.
2. Show sample images.
3. Overlay a grid so useful ROIs can be selected.
4. Define ROIs as rectangles.
5. Median-bin images to reduce noise.
6. Extract ROI medians, means, and percentiles over time.
7. Plot ratios and differences that should be stable to exposure changes.

In [ ]:
from pathlib import Path
import math

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import garage_monitor as gm

plt.rcParams["figure.figsize"] = (12, 7)
plt.rcParams["image.cmap"] = "gray"

config_path = Path("garage_config.yml")
if not config_path.exists():
    config_path = Path("/Users/grant/tmp/garage/garage_config.yml")

config = gm.load_config(config_path)
data_path = config["data_path"]
bin_factor = config["bin_factor"]
cache_dir_name = config["cache_dir_name"]
rois = config["rois"]

data_path


## Load Image Paths And Timestamps


In [ ]:
images, discovery_info = gm.discover_images(
    data_path,
    bin_factor=bin_factor,
    force_rescan=config.get("image_discovery", {}).get("force_rescan_images", False),
    cache_dir_name=cache_dir_name,
)

print(f"Feature cache used for discovery: {discovery_info['feature_cache_used_for_discovery']}")
latest_known_ts = discovery_info["latest_cached_filename_timestamp"]
if latest_known_ts is not None:
    print(f"Latest cached filename timestamp: {latest_known_ts:%Y-%m-%d %H:%M:%S}")
print(f"Top-level source files checked for readability: {discovery_info['top_level_source_files_checked']:,}")
print(f"New readable source images found: {discovery_info['new_readable_source_images']:,}")
print(f"Total image records available: {discovery_info['total_image_records']:,}")

if discovery_info["skipped_files"]:
    print(f"Skipped {len(discovery_info['skipped_files']):,} candidate image files")
    display(pd.DataFrame(discovery_info["skipped_files"]).head(20))

if images.empty:
    raise ValueError(f"No image records available under {data_path}")

images[gm.IMAGE_COLUMNS].head()


In [ ]:
images.tail()


## Sample Images


In [ ]:
idxs = gm.sample_indices(len(images), k=12)

ncols = 4
nrows = math.ceil(len(idxs) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4 * nrows), squeeze=False)
for ax, idx in zip(axes.ravel(), idxs):
    row = images.iloc[idx]
    gm.show_image(
        row.path,
        bin_factor=bin_factor,
        ax=ax,
        title=row.timestamp.strftime("%Y-%m-%d %H:%M"),
        cache_dir_name=cache_dir_name,
    )
for ax in axes.ravel()[len(idxs):]:
    ax.set_axis_off()
plt.tight_layout()


## Pick ROIs With A Grid

In [ ]:
grid_image_idx = len(images)-10
grid_step = 25

row = images.iloc[grid_image_idx]
fig, ax = plt.subplots(figsize=(14, 9))
gm.show_grid(row.path, bin_factor=bin_factor, step=grid_step, major_every=4, ax=ax, cache_dir_name=cache_dir_name)
ax.set_title(f"{row.timestamp:%Y-%m-%d %H:%M} | row {grid_image_idx} | {row.filename}")
plt.show()

## Define Candidate ROIs

Edit `garage_config.yml`, then rerun from the setup cell. ROI coordinates are in the median-binned image.


In [ ]:
fig, ax = plt.subplots(figsize=(14, 9))
gm.draw_rois(images.iloc[grid_image_idx].path, rois, bin_factor=bin_factor, ax=ax, cache_dir_name=cache_dir_name)
plt.show()


## Extract ROI Statistics


In [ ]:
feature_config = config.get("features", {})
features, feature_info = gm.build_features(
    images,
    rois,
    data_path,
    bin_factor=bin_factor,
    force_recompute=feature_config.get("force_recompute_features", False),
    max_images=feature_config.get("max_images"),
    cache_dir_name=cache_dir_name,
)

print(f"Feature cache: {feature_info['feature_cache_status']}")
print(f"Feature cache path: {feature_info['feature_cache_path']}")
print(f"Cached feature rows: {feature_info['cached_feature_rows']:,}")
print(f"Images needing feature extraction: {feature_info['images_needing_feature_extraction']:,} / {len(images):,}")
print(f"Extracted new features from {feature_info['extracted_new_features']:,} images")
print(f"Total feature rows available: {feature_info['total_feature_rows']:,}")
print(f"Binned cache: {feature_info['binned_cache_hits']:,} hits, {feature_info['binned_cache_misses']:,} misses")

if feature_info["binned_cache_warnings"]:
    print(f"Binned cache warnings: {len(feature_info['binned_cache_warnings']):,}")
    display(pd.DataFrame(feature_info["binned_cache_warnings"]).drop_duplicates().head(20))
if feature_info["feature_failures"]:
    print(f"Skipped {len(feature_info['feature_failures']):,} images during feature extraction")
    display(pd.DataFrame(feature_info["feature_failures"]).head(20))

features.head()


## Plotting Helpers


In [ ]:
def available_columns(columns, df=None):
    df = features if df is None else df
    return [c for c in columns if c in df.columns]


def plot_feature_timeseries(columns, title=None):
    columns = available_columns(columns)
    if not columns:
        print("No requested columns are available to plot.")
        return []

    fig, axes = plt.subplots(len(columns), 1, figsize=(14, 2.0 * len(columns)), sharex=True)
    if len(columns) == 1:
        axes = [axes]
    for ax, col in zip(axes, columns):
        ax.plot(features["timestamp"], features[col], marker=".", ms=2, lw=0.8)
        ax.set_ylabel(col)
        ax.grid(True, alpha=0.25)
    if title:
        fig.suptitle(title, y=1.01)
    plt.tight_layout()
    return columns


def plot_hour_scatter(plot_col):
    if plot_col not in features.columns:
        print(f"Column not found: {plot_col}")
        return
    tmp = features.copy()
    tmp["hour"] = pd.to_datetime(tmp["timestamp"]).dt.hour + pd.to_datetime(tmp["timestamp"]).dt.minute / 60

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.scatter(tmp["hour"], tmp[plot_col], s=8, alpha=0.5)
    ax.set_xlabel("hour of day")
    ax.set_ylabel(plot_col)
    ax.grid(True, alpha=0.25)
    plt.show()


def review_feature_extremes(review_col, review_n=6):
    if review_col not in features.columns:
        print(f"Column not found: {review_col}")
        return

    review = pd.concat(
        [features.nsmallest(review_n, review_col), features.nlargest(review_n, review_col)],
        ignore_index=True,
    ).drop_duplicates("image_key" if "image_key" in features.columns else "filename")
    review = review.sort_values("timestamp", ascending=False)

    if review.empty:
        print(f"No rows available for {review_col}.")
        return

    ncols = 4
    nrows = math.ceil(len(review) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4 * nrows), squeeze=False)
    for ax, (_, row) in zip(axes.ravel(), review.iterrows()):
        gm.draw_rois(row.path, rois, bin_factor=bin_factor, ax=ax, cache_dir_name=cache_dir_name)
        title = f"{row.timestamp:%Y-%m-%d %H:%M}"
        title += chr(10)
        title += f"{review_col}={row[review_col]:.3g}"
        ax.set_title(title)
    for ax in axes.ravel()[len(review):]:
        ax.set_axis_off()
    plt.tight_layout()


def plot_threshold_guess(feature_col, guess_col, threshold=None, higher_means_true=True):
    if feature_col not in features.columns:
        print(f"Column not found: {feature_col}")
        return None

    if threshold is None:
        threshold = features[feature_col].median()

    features[guess_col] = features[feature_col] > threshold if higher_means_true else features[feature_col] < threshold

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(features["timestamp"], features[feature_col], marker=".", ms=2, lw=0.8)
    ax.axhline(threshold, color="tab:red", lw=1.5, label=f"threshold={threshold:.3g}")
    ax.set_ylabel(feature_col)
    ax.grid(True, alpha=0.25)
    ax.legend()
    plt.show()

    display(features[["timestamp", "filename", feature_col, guess_col]].head())
    return threshold


def review_threshold_images(feature_col, guess_col, threshold, higher_means_true=True, n=12):
    if feature_col not in features.columns or guess_col not in features.columns:
        print(f"Missing {feature_col} or {guess_col}.")
        return

    mask = features[feature_col] > threshold if higher_means_true else features[feature_col] < threshold
    near = features[mask].copy()
    if higher_means_true:
        near = near.nsmallest(n, feature_col)
    else:
        near = near.nlargest(n, feature_col)
    near = near.sort_values("timestamp", ascending=False)

    if near.empty:
        print(f"No images match threshold={threshold:.3g} for {feature_col}.")
        return

    ncols = 4
    nrows = math.ceil(len(near) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4 * nrows), squeeze=False)
    for ax, (_, row) in zip(axes.ravel(), near.iterrows()):
        gm.draw_rois(row.path, rois, bin_factor=bin_factor, ax=ax, cache_dir_name=cache_dir_name)
        title = f"{row.timestamp:%Y-%m-%d %H:%M}"
        title += chr(10)
        title += f"{feature_col}={row[feature_col]:.3g}, guess={row[guess_col]}"
        ax.set_title(title)
    for ax in axes.ravel()[len(near):]:
        ax.set_axis_off()
    plt.tight_layout()


## Garage Door Signals


In [ ]:
garage_columns = [
    "door_median",
    "gap_or_outside_median",
    "wall_median",
    "door_to_wall_median",
    "gap_to_wall_median",
    "gap_p90_to_wall_median",
    "gap_p95_to_door_median",
]
garage_columns = plot_feature_timeseries(garage_columns, title="Garage door candidate signals")


In [ ]:
garage_rule = gm.threshold_rules_from_config(config)["garage_door_open"]
garage_plot_col = garage_rule.feature if garage_rule.feature in features.columns else garage_columns[-1]
plot_hour_scatter(garage_plot_col)


In [ ]:
garage_review_col = garage_plot_col
review_feature_extremes(garage_review_col, review_n=6)


In [ ]:
garage_threshold_col = garage_review_col
garage_higher_means_open = garage_rule.higher_means_true

_ = plot_threshold_guess(
    garage_threshold_col,
    guess_col="garage_door_open_guess",
    threshold=garage_rule.threshold,
    higher_means_true=garage_higher_means_open,
)


In [ ]:
review_threshold_images(
    garage_threshold_col,
    guess_col="garage_door_open_guess",
    threshold=garage_rule.threshold,
    higher_means_true=garage_higher_means_open,
    n=12,
)


## Car Presence Signals


In [ ]:
car_columns = [
    "car_window_median",
    "car_door_median",
    "wall_median",
    "car_window_to_car_door_median",
    "car_door_to_car_window_median",
    "car_window_to_wall_median",
    "car_door_to_wall_median",
    "car_door_minus_car_window_median",
    "car_window_p10_to_car_door_median",
    "car_window_to_car_door_p90",
    "car_window_p10_to_wall_median",
    "car_door_p90_to_wall_median",
]
car_columns = plot_feature_timeseries(car_columns, title="Car presence candidate signals")


In [ ]:
car_rule = gm.threshold_rules_from_config(config)["car_present"]
car_plot_col = car_rule.feature if car_rule.feature in features.columns else car_columns[-1]
plot_hour_scatter(car_plot_col)


In [ ]:
car_review_col = car_plot_col
review_feature_extremes(car_review_col, review_n=6)


In [ ]:
car_threshold_col = car_review_col
car_higher_means_present = car_rule.higher_means_true

_ = plot_threshold_guess(
    car_threshold_col,
    guess_col="car_present_guess",
    threshold=car_rule.threshold,
    higher_means_true=car_higher_means_present,
)


In [ ]:
# show when car absent
review_threshold_images(
    car_threshold_col,
    guess_col="car_present_guess",
    threshold=car_rule.threshold,
    higher_means_true=not car_higher_means_present,
    n=12,
)


## Latest Automation Output


In [ ]:
state, _, _ = gm.state_from_config(config)
state
